# 🍎 Générateur d'exécutable macOS — Text Saver App

Ce notebook crée un fichier `.app` macOS avec :
- **Sélecteur de dossier** pour choisir l'emplacement de sauvegarde
- **Zone de texte** pour saisir le contenu
- **Bouton Sauvegarder** qui écrit le texte dans un fichier `.txt`
- Compilé avec **PyInstaller** en application fenêtrée (sans console)


## 1. 📦 Importation des bibliothèques & vérification de PyInstaller

In [1]:
import os
import sys
import subprocess
from pathlib import Path

# ── Vérification / installation de PyInstaller ──────────────────────────────
try:
    import PyInstaller
    print(f"✅ PyInstaller déjà installé — version {PyInstaller.__version__}")
except ImportError:
    print("⏳ PyInstaller non trouvé, installation en cours…")
    subprocess.run([sys.executable, "-m", "pip", "install", "pyinstaller"], check=True)
    import PyInstaller
    print(f"✅ PyInstaller installé — version {PyInstaller.__version__}")

# Dossier de travail = dossier du notebook
WORK_DIR = Path().resolve()
print(f"📁 Dossier de travail : {WORK_DIR}")


⏳ PyInstaller non trouvé, installation en cours…
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 2.6 MB/s  0:00:0036m-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pyinstaller] [pyinstaller]hooks-contrib]
✅ PyInstaller installé — version 6.19.0
📁 Dossier de travail : /Users/grtk/Documents/GitHub/Pandas


## 2. 🖊️ Création du script Python source (`app.py`)

In [ ]:
APP_SOURCE = WORK_DIR / "app.py"

source_code = '''\
import tkinter as tk
from tkinter import filedialog, messagebox
import os

# ── Callbacks ────────────────────────────────────────────────────────────────
def browse_path():
    """Ouvre un dialogue de sélection de dossier."""
    folder = filedialog.askdirectory(title="Choisir un dossier de sauvegarde")
    if folder:
        path_var.set(folder)

def save_file():
    """Sauvegarde le contenu de la textbox dans un fichier .txt."""
    folder = path_var.get().strip()
    content = text_box.get("1.0", tk.END).strip()

    if not folder:
        messagebox.showwarning("Dossier manquant", "Veuillez sélectionner un dossier de sauvegarde.")
        return
    if not os.path.isdir(folder):
        messagebox.showerror("Dossier invalide", f"Le dossier sélectionné n'existe pas :\\n{folder}")
        return
    if not content:
        messagebox.showwarning("Texte vide", "La zone de texte est vide. Rien à sauvegarder.")
        return

    # Demander un nom de fichier
    file_path = filedialog.asksaveasfilename(
        initialdir=folder,
        title="Enregistrer sous…",
        defaultextension=".txt",
        filetypes=[("Fichiers texte", "*.txt"), ("Tous les fichiers", "*.*")],
    )
    if not file_path:
        return  # annulé par l'utilisateur

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)

    messagebox.showinfo("Sauvegardé ✓", f"Fichier enregistré :\\n{file_path}")

# ── Interface ─────────────────────────────────────────────────────────────────
root = tk.Tk()
root.title("Text Saver")
root.geometry("520x400")
root.resizable(True, True)
root.configure(padx=16, pady=12)

# ── Sélecteur de dossier ──────────────────────────────────────────────────────
path_var = tk.StringVar(value=os.path.expanduser("~"))

frame_path = tk.LabelFrame(root, text=" 📁  Dossier de sauvegarde ", padx=8, pady=6)
frame_path.pack(fill=tk.X, pady=(0, 10))

entry_path = tk.Entry(frame_path, textvariable=path_var, font=("Helvetica", 11))
entry_path.pack(side=tk.LEFT, fill=tk.X, expand=True, padx=(0, 6))

btn_browse = tk.Button(frame_path, text="Parcourir…", command=browse_path, width=10)
btn_browse.pack(side=tk.RIGHT)

# ── Zone de texte ─────────────────────────────────────────────────────────────
frame_text = tk.LabelFrame(root, text=" ✏️  Texte ", padx=8, pady=6)
frame_text.pack(fill=tk.BOTH, expand=True, pady=(0, 10))

scrollbar = tk.Scrollbar(frame_text)
scrollbar.pack(side=tk.RIGHT, fill=tk.Y)

text_box = tk.Text(
    frame_text,
    font=("Helvetica", 12),
    wrap=tk.WORD,
    yscrollcommand=scrollbar.set,
    relief=tk.FLAT,
    bd=2,
)
text_box.pack(fill=tk.BOTH, expand=True)
scrollbar.config(command=text_box.yview)

# ── Bouton Sauvegarder ────────────────────────────────────────────────────────
btn_save = tk.Button(
    root,
    text="💾  Sauvegarder",
    command=save_file,
    font=("Helvetica", 13, "bold"),
    height=2,
    bg="#2d6a4f",
    fg="white",
    activebackground="#1b4332",
    activeforeground="white",
    relief=tk.FLAT,
    cursor="hand2",
)
btn_save.pack(fill=tk.X)

root.mainloop()
'''

APP_SOURCE.write_text(source_code, encoding="utf-8")

if APP_SOURCE.exists():
    print(f"✅ Script créé : {APP_SOURCE}")
    print("─" * 50)
    print(source_code)
else:
    print("❌ Échec de la création du script")


✅ Script créé : /Users/grtk/Documents/GitHub/Pandas/app.py
──────────────────────────────────────────────────
import tkinter as tk
from tkinter import messagebox

def on_ok():
    messagebox.showinfo("Message", "Bienvenue !")

root = tk.Tk()
root.title("Bienvenue App")
root.geometry("260x120")
root.resizable(False, False)

# Centrer la fenêtre
root.eval("tk::PlaceWindow . center")

label = tk.Label(root, text="Cliquez sur OK pour continuer", pady=16)
label.pack()

btn = tk.Button(root, text="OK", width=12, height=2, command=on_ok)
btn.pack()

root.mainloop()



## 3. ⚙️ Génération du fichier de configuration PyInstaller (`.spec`)

In [ ]:
SPEC_FILE = WORK_DIR / "app.spec"

spec_content = f"""\
# -*- mode: python ; coding: utf-8 -*-

a = Analysis(
    ['{APP_SOURCE}'],
    pathex=['{WORK_DIR}'],
    binaries=[],
    datas=[],
    hiddenimports=['tkinter', 'tkinter.filedialog', 'tkinter.messagebox'],
    hookspath=[],
    hooksconfig={{}},
    runtime_hooks=[],
    excludes=[],
    noarchive=False,
)

pyz = PYZ(a.pure)

exe = EXE(
    pyz,
    a.scripts,
    a.binaries,
    a.datas,
    name='TextSaver',
    debug=False,
    bootloader_ignore_signals=False,
    strip=False,
    upx=False,
    console=False,       # ← pas de fenêtre Terminal
    target_arch=None,
    codesign_identity=None,
    entitlements_file=None,
)

app = BUNDLE(
    exe,
    name='TextSaver.app',
    bundle_identifier='com.local.textsaver',
    info_plist={{
        'NSHighResolutionCapable': True,
        'CFBundleShortVersionString': '1.0.0',
        'CFBundleName': 'TextSaver',
    }},
)
"""

SPEC_FILE.write_text(spec_content, encoding="utf-8")

if SPEC_FILE.exists():
    print(f"✅ Fichier .spec créé : {SPEC_FILE}")
else:
    print("❌ Échec de la création du .spec")


✅ Fichier .spec créé : /Users/grtk/Documents/GitHub/Pandas/app.spec


## 4. 🔨 Compilation en exécutable macOS (`.app`)

In [4]:
print("⏳ Compilation en cours… (peut prendre 30–60 s)")
print("─" * 50)

result = subprocess.run(
    [
        sys.executable, "-m", "PyInstaller",
        "--clean",
        "--noconfirm",
        "--distpath", str(WORK_DIR / "dist"),
        "--workpath", str(WORK_DIR / "build"),
        str(SPEC_FILE),
    ],
    capture_output=True,
    text=True,
    cwd=str(WORK_DIR),
)

# Affichage des logs
if result.stdout:
    for line in result.stdout.splitlines():
        if any(kw in line for kw in ["INFO", "WARNING", "ERROR", "Building", "completed"]):
            print(line)

if result.returncode == 0:
    print("\n✅ Compilation terminée avec succès !")
else:
    print("\n❌ Erreur de compilation :")
    print(result.stderr[-3000:])   # derniers 3000 chars pour ne pas saturer l'output


⏳ Compilation en cours… (peut prendre 30–60 s)
──────────────────────────────────────────────────

✅ Compilation terminée avec succès !


## 5. ✅ Vérification & lancement de l'application

In [ ]:
APP_BUNDLE = WORK_DIR / "dist" / "TextSaver.app"

if APP_BUNDLE.exists():
    print(f"🎉 Application créée avec succès !")
    print(f"📦 Chemin : {APP_BUNDLE.resolve()}")
    print(f"📏 Taille : {sum(f.stat().st_size for f in APP_BUNDLE.rglob('*') if f.is_file()) / 1_048_576:.1f} Mo")

    # ── Lancer l'app directement depuis le notebook ──
    launch = input("\n▶️  Lancer l'application maintenant ? (o/n) : ").strip().lower()
    if launch in ("o", "oui", "y", "yes"):
        subprocess.Popen(["open", str(APP_BUNDLE)])
        print("✅ Application lancée !")
    else:
        print(f"ℹ️  Pour lancer manuellement : double-cliquez sur {APP_BUNDLE.name} dans le Finder")
        print(f"   ou exécutez : open \"{APP_BUNDLE}\"")
else:
    print("❌ Le fichier .app est introuvable — vérifiez les logs de compilation (cellule 4).")
    print(f"   Chemin attendu : {APP_BUNDLE}")


🎉 Application créée avec succès !
📦 Chemin : /Users/grtk/Documents/GitHub/Pandas/dist/Bienvenue.app
📏 Taille : 9.7 Mo
✅ Application lancée !
